In [ ]:
# Imports
import os, re, unicodedata
from pathlib import Path
from calendar import month_abbr

import numpy as np
import pandas as pd

# File paths
NBA_PATH = '/content/drive/MyDrive/nba-draft-sucess/stat_scraper/nba_player_stats.csv'
COMBINE_PATH = '/content/drive/MyDrive/nba-draft-sucess/stat_scraper/nba_combine.csv'
OUT_DIR = '/content/drive/MyDrive/nba-draft-sucess/final_master'
OUT_FILE = 'nba_stats_with_labels.csv'

# Make sure the output folder exists
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:
# Mount Google Drive
if os.path.exists("/content/drive"):
    try:
        !fusermount -u /content/drive 2>/dev/null
        !rm -rf /content/drive
    except Exception as e:
        print("Unmount warning:", e)

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
def to_num(s):
    """Convert to numeric; return NaN on errors."""
    return pd.to_numeric(s, errors="coerce")

def strip_accents(s: str) -> str:
    """Remove accents from a string."""
    return ''.join(ch for ch in unicodedata.normalize('NFKD', s) if not unicodedata.combining(ch))

# Suffixes to ignore in names
SUFFIXES = {"jr", "jr.", "sr", "sr.", "ii", "iii", "iv", "v"}

def clean_tokens(tokens):
    """Lowercase, strip punctuation, remove suffixes."""
    out = []
    for t in tokens:
        t = strip_accents(t.strip().lower())
        t = re.sub(r"[^\w\s-]", "", t)
        if t and t not in SUFFIXES:
            out.append(t)
    return out

def make_join_key_from_first_last(raw: str) -> str:
    """NBA side: 'First [Middle] Last' -> 'first last'."""
    if not isinstance(raw, str) or not raw.strip():
        return ""
    toks = clean_tokens(raw.split())
    if len(toks) >= 2:
        first = toks[0]
        last = " ".join(toks[1:])
        key = f"{first} {last}"
    else:
        key = " ".join(toks)
    return re.sub(r"\s+", " ", key).strip()

def make_join_key_from_last_comma_first(raw: str) -> str:
    """Combine side: 'Last, First [Middle]' -> 'first last'."""
    if not isinstance(raw, str) or not raw.strip():
        return ""
    s = raw.strip()
    if "," in s:
        last, firsts = s.split(",", 1)
        last_tokens = clean_tokens(last.split())
        first_tokens = clean_tokens(firsts.split())
        if not first_tokens or not last_tokens:
            return ""
        key = f"{first_tokens[0]} {' '.join(last_tokens)}"
    else:
        key = make_join_key_from_first_last(raw)
    return re.sub(r"\s+", " ", key).strip()


In [ ]:
# Map 'Jan'...'Dec' (lowercased) to month index 1..12
MON_MAP = {m.lower(): i for i, m in enumerate(month_abbr) if m}

def parse_season_start_year(season_str):
    """
    Parse '2016-17' or '2016-2017' -> 2016.
    Parse 'Jan-00' -> season start year (handles Jan–Jun rolling to previous start).
    """
    if not isinstance(season_str, str) or not season_str.strip():
        return pd.NA
    s = season_str.strip()

    # '2016-17' or '2016-2017'
    m = re.match(r"^\s*(\d{4})\s*-\s*(\d{2,4})\s*$", s)
    if m:
        return int(m.group(1))

    # 'Jan-00'
    m = re.match(r"^\s*([A-Za-z]{3})\s*-\s*(\d{2})\s*$", s)
    if m:
        mon = m.group(1).lower()
        yy = int(m.group(2))
        year = 2000 + yy if yy <= 49 else 1900 + yy
        month = MON_MAP.get(mon)
        # If the month is Jan–Jun, treat it as the previous season start
        return (year - 1) if month and 1 <= month <= 6 else year

    return pd.NA

def primary_pos(s):
    """Return the first position token (e.g., 'SG' from 'SG/SF')."""
    if not isinstance(s, str) or not s:
        return "UNK"
    return re.split(r"[/\- ,]", s.upper())[0] or "UNK"

def z_by_group(frame, group_col, col):
    """
    Z-score within group (e.g., by position).
    Returns 0 if std=0 or column missing.
    """
    if col not in frame.columns:
        return pd.Series(np.zeros(len(frame)), index=frame.index)

    def z(x):
        mu = x.mean(skipna=True)
        sd = x.std(ddof=0, skipna=True)
        if not np.isfinite(sd) or sd == 0:
            return pd.Series(np.zeros(len(x)), index=x.index)
        return (x - mu) / sd

    return frame.groupby(group_col)[col].transform(z).fillna(0.0)


In [ ]:
# Load data + strip whitespace
nba = pd.read_csv(NBA_PATH, dtype=str, keep_default_na=False)
combine = pd.read_csv(COMBINE_PATH, dtype=str, keep_default_na=False)

for df in (nba, combine):
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].str.strip()

print(f"NBA rows: {len(nba):,}, cols: {len(nba.columns)}")
print(f"Combine rows: {len(combine):,}, cols: {len(combine.columns)}")


In [ ]:
# Filter to NBA league only

if "Lg" in nba.columns:
    nba["Lg"] = nba["Lg"].str.upper()
    nba = nba[nba["Lg"] == "NBA"].copy()


In [ ]:
# Find the best NBA name column
name_col_nba = next((c for c in ["PlayerName", "PlayerName_per_game", "PlayerName_adv", "Player"] if c in nba.columns), None)
if not name_col_nba:
    raise KeyError("Could not find a PlayerName column in NBA CSV.")
nba["join_key"] = nba[name_col_nba].apply(make_join_key_from_first_last)

# Parse season start, drop bad rows, cast to int
nba["SeasonStart"] = nba["Season"].apply(parse_season_start_year)
nba = nba.dropna(subset=["SeasonStart"]).copy()
nba["SeasonStart"] = nba["SeasonStart"].astype(int)

# Combine file must have 'PLAYER' like 'Last, First'
if "PLAYER" not in combine.columns:
    raise KeyError("Combine file must have a 'PLAYER' column like 'Last, First'.")
combine["join_key"] = combine["PLAYER"].apply(make_join_key_from_last_comma_first)


In [ ]:
# Played from multiple possible columns

G_candidates  = [c for c in ["G_adv", "G_per_game", "G"] if c in nba.columns]
MP_candidates = [c for c in ["MP_adv", "MP"] if c in nba.columns]

# Any available "Games"
G_any = None
for c in G_candidates:
    vals = to_num(nba[c])
    G_any = vals if G_any is None else G_any.fillna(vals)
if G_any is None:
    G_any = pd.Series(0, index=nba.index, dtype=float)

# Any available "Minutes"
MP_any = None
for c in MP_candidates:
    vals = to_num(nba[c])
    MP_any = vals if MP_any is None else MP_any.fillna(vals)
if MP_any is None and "MP_per_game" in nba.columns:
    MP_any = to_num(nba["MP_per_game"]) * to_num(G_any)
if MP_any is None:
    MP_any = pd.Series(0, index=nba.index, dtype=float)

nba["_played_flag"] = (G_any.fillna(0) >= 1) | (MP_any.fillna(0) > 0)


In [ ]:
# Count seasons by PlayerID and filter to >=8

pid_col = next((c for c in ["PlayerID", "PlayerID_per_game", "PlayerID_adv"] if c in nba.columns), None)
if not pid_col:
    raise KeyError("Could not find a PlayerID column in NBA CSV.")

played = nba[nba["_played_flag"]].drop_duplicates(subset=[pid_col, "SeasonStart"])
season_counts = played.groupby(pid_col)["SeasonStart"].nunique().rename("nba_seasons_played").reset_index()

eligible_ids = set(season_counts.loc[season_counts["nba_seasons_played"] >= 8, pid_col])
nba8 = nba[nba[pid_col].isin(eligible_ids)].copy()

print(f"Players with >=8 NBA seasons (by PlayerID): {season_counts['nba_seasons_played'].ge(8).sum()}")


In [ ]:
# Match only players present in both datasets

nba_keys = nba8[["join_key"]].drop_duplicates()
combine_keys = combine[["join_key"]].drop_duplicates()
both_keys = pd.merge(nba_keys, combine_keys, on="join_key", how="inner")

print("Matched players (NBA>=8 ∩ Combine):", len(both_keys))

nba8 = pd.merge(nba8, both_keys, on="join_key", how="inner")
combine_in = pd.merge(combine, both_keys, on="join_key", how="inner")


In [ ]:
# Career averages (per join_key)
ADV_KEEP = [
    "PER","TS%","3PAr","FTr","ORB%","DRB%","TRB%","AST%","STL%","BLK%","TOV%","USG%",
    "OWS","DWS","WS","WS/48","OBPM","DBPM","BPM","VORP"
]
PERGAME_SRC = [
    "G_per_game","GS_per_game","MP_per_game","FG","FGA","FG%","3P","3PA","3P%","2P","2PA","2P%",
    "eFG%","FT","FTA","FT%","ORB","DRB","TRB","AST","STL","BLK","TOV","PF","PTS"
]

# Convert numeric columns
for c in ADV_KEEP + PERGAME_SRC:
    if c in nba8.columns:
        nba8[c] = to_num(nba8[c])

# Average by player
keep_cols = [c for c in ADV_KEEP + PERGAME_SRC if c in nba8.columns]
nba_avgs = nba8.groupby("join_key")[keep_cols].mean(numeric_only=True).reset_index()

# Rename common per-game columns
nba_avgs.rename(columns={"G_per_game":"GP","GS_per_game":"GS","MP_per_game":"MP"}, inplace=True)


In [ ]:
#  Combine metadata (name, POS, measurables)

COMBINE_KEEP = [
    "HGT","WGT","BMI","BF","WNGSPN","STNDRCH","HANDL","HANDW",
    "STNDVERT","LPVERT","LANE","SHUTTLE","SPRINT","BENCH","BAR","PAN","PBHGT","PDHGT"
]

comb_meta_cols = ["PLAYER","POS"] + [c for c in COMBINE_KEEP if c in combine_in.columns]

# Take the first non-null value within each player group
comb_meta = (
    combine_in.groupby("join_key")[comb_meta_cols]
    .agg(lambda x: x.dropna().iloc[0] if len(x.dropna()) else np.nan)
    .reset_index()
)

def player_from_last_first(s):
    """Turn 'Last, First' into 'First Last'."""
    if isinstance(s, str) and "," in s:
        last, first = s.split(",", 1)
        return f"{first.strip()} {last.strip()}"
    return ""

comb_meta["Player"] = comb_meta["PLAYER"].apply(player_from_last_first)
mask_empty = comb_meta["Player"].eq("")
comb_meta.loc[mask_empty, "Player"] = comb_meta.loc[mask_empty, "join_key"].str.title()


In [ ]:
#  Merge + remove duplicate columns

final_df = pd.merge(
    comb_meta[["join_key","Player","POS"] + [c for c in COMBINE_KEEP if c in comb_meta.columns]],
    nba_avgs,
    on="join_key",
    how="inner"
)

# Drop duplicate columns if any
final_df = final_df.loc[:, ~final_df.columns.duplicated()].copy()


In [ ]:
# Order columns and format numbers

final_order = (
    ["Player","POS"] +
    [c for c in COMBINE_KEEP if c in final_df.columns] +
    [c for c in ADV_KEEP if c in final_df.columns] +
    [c for c in ["GP","GS","MP","FG","FGA","FG%","3P","3PA","3P%","2P","2PA","2P%","eFG%","FT","FTA","FT%","ORB","DRB","TRB","AST","STL","BLK","TOV","PF","PTS"] if c in final_df.columns]
)
final_order = [c for c in final_order if c in final_df.columns]
final_df = final_df[final_order].copy()

# Ensure numeric types
for c in final_df.columns:
    if c in ["Player","POS"]:
        continue
    final_df[c] = to_num(final_df[c])

# Round: integers for GP/GS; 2 decimals elsewhere
for c in final_df.columns:
    if c in ["GP","GS"]:
        final_df[c] = final_df[c].round().astype("Int64")
    elif c not in ["Player","POS"]:
        final_df[c] = final_df[c].round(2)

print("Players after merge & averaging:", len(final_df))


In [ ]:
# Scoring and label assignment

final_df["POS_PRIMARY"] = final_df["POS"].apply(primary_pos)

# Make sure needed numeric fields are numbers
NUMS = ["BPM","WS/48","VORP","WS","TS%","eFG%","TOV%","FT%","DBPM","STL%","BLK%","DRB%","AST%","USG%","GP","MP","GS"]
for c in NUMS:
    if c in final_df.columns:
        final_df[c] = to_num(final_df[c])

def zpos(col):
    """Helper: z-score of a column within POS_PRIMARY groups."""
    return z_by_group(final_df, "POS_PRIMARY", col)

# Component scores
impact = zpos("BPM") + 0.70*zpos("WS/48") + 0.60*zpos("VORP") + 0.40*zpos("WS")
eff    = 0.70*zpos("TS%") + 0.30*zpos("eFG%") - 0.30*zpos("TOV%") + 0.20*zpos("FT%")
deff   = 0.80*zpos("DBPM") + 0.30*zpos("STL%") + 0.30*zpos("BLK%") + 0.20*zpos("DRB%")
play   = 0.70*zpos("AST%") + 0.30*zpos("USG%") - 0.30*zpos("TOV%")
dur    = 0.50*zpos("GP")   + 0.35*zpos("MP")   + 0.15*zpos("GS")

# Save components and composite
final_df["_ImpactScore"] = impact
final_df["_EffScore"]    = eff
final_df["_DefScore"]    = deff
final_df["_PlayScore"]   = play
final_df["_DurScore"]    = dur

final_df["CompositeScore"] = (0.35*impact + 0.20*eff + 0.15*deff + 0.10*play + 0.20*dur)

# Percentile thresholds for labels
valid = final_df["CompositeScore"].astype(float)
p_super   = np.nanpercentile(valid, 95) if len(valid) else np.nan
p_star    = np.nanpercentile(valid, 80) if len(valid) else np.nan
p_starter = np.nanpercentile(valid, 50) if len(valid) else np.nan
p_role    = np.nanpercentile(valid, 20) if len(valid) else np.nan

# Auto rules for extreme cases (dict keys are arbitrary names, not column names)
AUTO_BUST  = {"BPM": -1.0, "WS48": 0.08, "GP": 30}
AUTO_SUPER = {"BPM": 5.0,  "WS48": 0.20}

def label_row(r):
    """Assign label based on composite + simple auto rules."""
    cs = r["CompositeScore"]
    if pd.isna(cs):
        return "Unknown"

    bpm  = r.get("BPM", np.nan)
    ws48 = r.get("WS/48", np.nan)
    gp   = r.get("GP", np.nan)

    # auto-superstar if BPM or WS/48 is very high
    if (pd.notna(bpm) and bpm > AUTO_SUPER["BPM"]) or (pd.notna(ws48) and ws48 > AUTO_SUPER["WS48"]):
        return "Superstar"

    # auto-bust if BPM, WS/48, and GP are all weak
    if (pd.notna(bpm) and bpm < AUTO_BUST["BPM"]) and (pd.notna(ws48) and ws48 < AUTO_BUST["WS48"]) and (pd.notna(gp) and gp < AUTO_BUST["GP"]):
        return "Bust"

    # percentile-based labels
    if cs >= p_super:   return "Superstar"
    if cs >= p_star:    return "Star"
    if cs >= p_starter: return "Starter"
    if cs >= p_role:    return "Role Player"
    return "Bust"

final_df["Label"] = final_df.apply(label_row, axis=1)


In [ ]:
# Remove not needed columns before saving

drop_cols = [
    # measurables
    "HGT","WGT","BMI","BF","WNGSPN","STNDRCH","HANDL","HANDW","STNDVERT","LPVERT",
    "LANE","SHUTTLE","SPRINT","BENCH","BAR","PAN","PBHGT","PDHGT",
    "POS_PRIMARY","_ImpactScore","_EffScore","_DefScore","_PlayScore","_DurScore","CompositeScore"
]

final_clean = final_df.drop(columns=[c for c in drop_cols if c in final_df.columns], errors="ignore")


In [ ]:
# Save results

save_path = f"{OUT_DIR}/{OUT_FILE}"
final_clean.to_csv(save_path, index=False)

print(f"Saved cleaned file → {save_path}")
print(f"Rows: {len(final_clean)}, Columns: {len(final_clean.columns)}")
print(final_clean["Label"].value_counts(dropna=False))
